In [9]:
import pandas as pd
import numpy as np
import os
import re
import xarray as xr
import json
from pathlib import Path

# ============= НАСТРОЙКИ =============
base_dir = 'C:/Users/Maks/Desktop/Jupyter/wind_profiles/+'
thunderbolts_base_dir = 'C:/Users/Maks/Desktop/Jupyter/thunderbolts_data/hdf'
satellite_dir = 'C:/Users/Maks/Desktop/Jupyter/satellite_data/TIDI'
filtered_satellite_dir = 'C:/Users/Maks/Desktop/Jupyter/output_filtered_satellite_data'
matching_files_dir = 'C:/Users/Maks/Desktop/Jupyter/output_full_data'
statistics_dir = 'C:/Users/Maks/Desktop/Jupyter/Statistics'

# Годы, которые нужно обработать
years = [2012, 2013, 2014, 2015, 2016, 2017, 2018]

# Функция извлечения номера кластера из имени файла
def extract_cluster_number(filename):
    match = re.search(r'TIDI_cluster(\d+)_', filename)
    if match:
        return int(match.group(1))
    return None

# Функция извлечения даты и времени из имени файла
def extract_datetime_from_filename(filename):
    match = re.search(r'_(\d{8})_(\d{6})\.png$', filename)
    if match:
        date_str = match.group(1)  # DDMMYYYY
        time_str = match.group(2)  # HHMMSS
        day = date_str[0:2]
        month = date_str[2:4]
        year = date_str[4:8]
        datetime_str = f"{year}-{month}-{day} {time_str[:2]}:{time_str[2:4]}:{time_str[4:6]}"
        return pd.to_datetime(datetime_str)
    return None

# Функция для получения всех PNG файлов за определенный год
def get_png_files_for_year(year):
    year_dir = os.path.join(base_dir, str(year))
    if not os.path.exists(year_dir):
        return []
    png_files = []
    for filename in os.listdir(year_dir):
        if filename.endswith('.png') and 'cluster' in filename:
            png_files.append(os.path.join(year_dir, filename))
    return png_files

# Функция загрузки данных о разрядах для года
def load_thunderbolts_data(year):
    """Загружает данные о разрядах для указанного года"""
    thunderbolts_path = os.path.join(thunderbolts_base_dir, f'{year}_thunderbolts_clastered.h5')
    if not os.path.exists(thunderbolts_path):
        return None
    try:
        data = pd.read_hdf(thunderbolts_path, 'strikes')
        return data
    except Exception:
        return None

# Функция загрузки данных о совпадениях для указанного года
def load_matching_data(year):
    matching_path = os.path.join(matching_files_dir, f'{year}_satellite_overpass_matching.h5')
    if not os.path.exists(matching_path):
        return None
    try:
        data = pd.read_hdf(matching_path, key='matches')
        return data
    except Exception:
        return None

# Функция для извлечения профиля ветра
def get_wind_profile(cluster_num, year, match_time, matching_data):
    try:
        if cluster_num not in matching_data.index:
            return None
        row = matching_data.loc[cluster_num]
        tidi_file_full = row['satellite_file_name']
        tidi_file = tidi_file_full.replace('.VEC', '')
        
        tidi_path = os.path.join(satellite_dir, str(year), tidi_file + '.VEC')
        
        if not os.path.exists(tidi_path):
            return None
        
        ds = xr.open_dataset(tidi_path, decode_timedelta=True)
        df_full = ds.to_dataframe().reset_index()
        ds.close()
        
        base_date = pd.Timestamp('1980-01-06 00:00:00')
        df_full['datetime'] = base_date + pd.to_timedelta(df_full['time'], unit='ns')
        
        df_full['time_diff'] = abs(df_full['datetime'] - match_time)
        closest_idx = df_full['time_diff'].idxmin()
        
        closest_nvec = df_full.loc[closest_idx, 'nvec']
        nvec_data = df_full[df_full['nvec'] == closest_nvec]
        
        nvec_data = nvec_data.sort_values('alt_retrieved')
        
        altitudes = []
        u_values = []
        v_values = []
        wind_speeds = []
        
        for _, row in nvec_data.iterrows():
            if pd.notna(row['u']) and pd.notna(row['v']):
                altitudes.append(round(row['alt_retrieved'], 1))
                u_values.append(round(row['u'], 1))
                v_values.append(round(row['v'], 1))
                wind_speeds.append(round(np.sqrt(row['u']**2 + row['v']**2), 1))
        
        if len(altitudes) == 0:
            return None
        
        wind_profile = {
            'altitudes': altitudes,
            'u': u_values,
            'v': v_values,
            'wind_speed': wind_speeds
        }
        
        return wind_profile
        
    except Exception:
        return None

# Функция для получения статистики по разрядам
def get_cluster_statistics(cluster_num, thunderbolts_data, match_time):
    
    cluster_strikes = thunderbolts_data[thunderbolts_data['clnb'] == cluster_num]
    
    if len(cluster_strikes) == 0:
        return None
    
    strikes_until_match = cluster_strikes[cluster_strikes.index <= match_time]
    
    if len(strikes_until_match) == 0:
        return None
    
    # 1. Мощность последнего разряда перед попаданием 
    last_strike = strikes_until_match.iloc[-1]
    last_flash_amp = last_strike['amp']
    
    # 2. Средняя мощность всех разрядов от начала до попадания (ПО МОДУЛЮ, округление до 1 знака)
    mean_power_until_match = round(strikes_until_match['amp'].abs().mean(), 1)
    
    # 3. Разряды за 15 минут до попадания
    time_15min_before = match_time - pd.Timedelta(minutes=15)
    strikes_15min = strikes_until_match[
        (strikes_until_match.index >= time_15min_before) & 
        (strikes_until_match.index < match_time)
    ]
    
    count_15min = len(strikes_15min)
    
    # 4. Средняя мощность за 15 минут до попадания (ПО МОДУЛЮ, округление до 1 знака)
    if count_15min > 0:
        mean_power_15min = round(strikes_15min['amp'].abs().mean(), 1)
    else:
        mean_power_15min = np.nan
    
    # 5. Количество всех разрядов от начала до попадания
    total_count_until_match = len(strikes_until_match)
    
    # 6. Время существования кластера до попадания (округление до 1 знака)
    cluster_lifetime = round((match_time - strikes_until_match.index.min()).total_seconds() / 60, 1)

    # Суммарная мощность всех разрядов до попадания
    total_power_until_match = strikes_until_match['amp'].abs().sum()

    # Суммарная мощность за 15 минут до попадания
    total_power_15min = strikes_15min['amp'].abs().sum()
    
    return {
        'cluster_number': cluster_num,
        'match_time': match_time,
        'last_flash_amp': last_flash_amp,
        'mean_power_until_match': mean_power_until_match,
        'mean_power_15min': mean_power_15min,
        'total_power_until_match': total_power_until_match,  
        'total_power_15min': total_power_15min,             
        'total_count_until_match': total_count_until_match,
        'count_15min': count_15min,
        'cluster_lifetime_minutes': cluster_lifetime
    }

# ============= ОСНОВНОЙ КОД =============

# Список для хранения всех записей
all_statistics = []

# Обрабатываем каждый год
for year in years:
    print(f"\n{'='*60}")
    print(f"Обработка {year} года")
    print(f"{'='*60}")
    
    # Загружаем данные о разрядах для года
    thunderbolts_data = load_thunderbolts_data(year)
    if thunderbolts_data is None:
        continue
    
    # Загружаем matching данные
    matching_data = load_matching_data(year)
    if matching_data is None:
        continue
    
    # Получаем все PNG файлы за этот год
    png_files = get_png_files_for_year(year)
    print(f"Найдено PNG файлов: {len(png_files)}")
    
    # Обрабатываем каждый PNG файл
    for png_path in png_files:
        filename = os.path.basename(png_path)
        
        # Извлекаем номер кластера
        cluster_num = extract_cluster_number(filename)
        if cluster_num is None:
            continue
        
        # Извлекаем дату и время
        match_time = extract_datetime_from_filename(filename)
        if match_time is None:
            continue
        
        # Получаем статистику по разрядам
        stats = get_cluster_statistics(cluster_num, thunderbolts_data, match_time)
        
        if stats is None:
            continue
        
        # Получаем профиль ветра
        wind_profile = get_wind_profile(cluster_num, year, match_time, matching_data)
        
        if wind_profile is None:
            continue
        
        # Сохраняем профили как списки (матрицы)
        stats['altitudes'] = wind_profile['altitudes']
        stats['u'] = wind_profile['u']
        stats['v'] = wind_profile['v']
        stats['wind_speed'] = wind_profile['wind_speed']
        
        # Добавляем в общий список
        all_statistics.append(stats)

# Создаем DataFrame со всей статистикой
if all_statistics:
    # Создаем DataFrame для сохранения
    df_save = pd.DataFrame(all_statistics)
    
    # Сохраняем профили в виде JSON-строк для удобства
    for col in ['altitudes', 'u', 'v', 'wind_speed']:
        if col in df_save.columns:
            df_save[col] = df_save[col].apply(lambda x: json.dumps(x))
    
    # Создаем директорию для статистики, если её нет
    os.makedirs(statistics_dir, exist_ok=True)
    
    # Сохраняем в CSV
    output_csv = os.path.join(statistics_dir, 'wind_statistics_all.csv')
    df_save.to_csv(output_csv, index=False, encoding='utf-8')
    print(f"\nДанные сохранены в {output_csv}")
    
else:
    print("\nНет данных для анализа")

print("\nОбработка завершена!")


Обработка 2012 года
Найдено PNG файлов: 5

Обработка 2013 года
Найдено PNG файлов: 14

Обработка 2014 года
Найдено PNG файлов: 24

Обработка 2015 года
Найдено PNG файлов: 8

Обработка 2016 года
Найдено PNG файлов: 9

Обработка 2017 года
Найдено PNG файлов: 1

Обработка 2018 года
Найдено PNG файлов: 1

Данные сохранены в C:/Users/Maks/Desktop/Jupyter/Statistics\wind_statistics_all.csv

Обработка завершена!


In [10]:
df_save

,cluster_number,match_time,last_flash_amp,mean_power_until_match,mean_power_15min,total_power_until_match,total_power_15min,total_count_until_match,count_15min,cluster_lifetime_minutes,altitudes,u,v,wind_speed
0,154,2012-07-15 22:42:27,1697.0,3556.6,2653.6,67576.0,13268.0,19,5,116.8,"[82.5, 85.0, 87.5, 90.0, 92.5, 95.0]","[44.9, 29.9, 35.7, 41.8, 65.4, 67.1]","[7.0, 57.3, 38.8, 26.5, 45.0, 52.2]","[45.5, 64.6, 52.7, 49.5, 79.4, 85.0]"
1,3967,2012-08-12 13:27:10,-8165.0,6968.3,8424.7,1031303.0,50548.0,148,6,267.1,"[82.5, 85.0, 87.5, 90.0, 92.5, 95.0, 97.5, 100...","[-50.4, -49.1, -15.7, 21.8, 44.5, 57.5, 78.0, ...","[47.1, 40.1, 31.8, 14.3, 13.6, 28.6, 6.5, -33....","[68.9, 63.4, 35.5, 26.1, 46.5, 64.2, 78.3, 97...."
2,4376,2012-08-17 08:12:31,5510.0,5510.0,5510.0,5510.0,5510.0,1,1,0.7,"[82.5, 85.0, 87.5, 90.0, 92.5, 95.0, 97.5, 100...","[-14.5, -3.0, 5.8, 18.8, 35.2, 52.0, 55.7, 32....","[-3.9, 21.3, 37.0, 19.9, -3.1, -20.6, -17.9, 2...","[15.0, 21.5, 37.5, 27.4, 35.3, 56.0, 58.5, 32...."
3,504,2012-07-18 02:31:02,3350.0,4848.6,2503.3,140608.0,7510.0,29,3,100.7,"[82.5, 85.0, 87.5, 90.0, 92.5, 95.0, 97.5, 100.0]","[-8.7, -4.8, 3.1, 19.3, 32.1, 34.4, 27.6, 14.1]","[7.1, 9.1, 17.9, 29.6, 38.0, 40.5, 37.2, 31.3]","[11.2, 10.3, 18.2, 35.4, 49.7, 53.1, 46.3, 34.4]"
4,1123,2013-06-06 14:54:45,8797.0,11697.4,19499.7,210554.0,58499.0,18,3,87.8,"[82.5, 85.0, 87.5, 90.0, 92.5, 95.0, 97.5, 100.0]","[-48.0, -29.7, -3.0, 12.7, 23.1, 27.7, 37.6, 3...","[-25.9, -6.0, 17.9, 8.3, -9.6, -8.0, 13.1, 10.1]","[54.5, 30.3, 18.2, 15.2, 25.0, 28.8, 39.8, 32.5]"
5,1249,2013-06-08 08:47:31,1820.0,6303.5,1820.0,12607.0,1820.0,2,1,19.5,"[85.0, 87.5, 90.0, 92.5, 95.0, 97.5, 100.0]","[-4.6, -22.0, -48.1, -28.3, -17.8, -23.8, -13.5]","[-0.2, -14.2, -40.3, -27.7, 1.1, 27.2, 70.6]","[4.6, 26.2, 62.8, 39.6, 17.8, 36.2, 71.8]"
6,1492,2013-06-10 15:51:08,4862.0,6958.1,6522.6,1022841.0,456582.0,147,70,37.4,"[82.5, 85.0, 87.5, 90.0, 92.5, 95.0, 97.5, 100...","[-36.4, -47.4, -44.3, -27.0, -8.5, 16.7, 45.9,...","[-26.8, -28.4, -0.6, 38.8, 66.5, 82.6, 87.7, 7...","[45.2, 55.2, 44.3, 47.3, 67.1, 84.3, 98.9, 101..."
7,1608,2013-06-11 14:26:38,-28690.0,11880.2,13629.1,142562.0,95404.0,12,7,61.0,"[82.5, 85.0, 87.5, 90.0, 92.5, 95.0, 97.5, 100...","[-33.6, -42.6, -38.5, -17.1, 17.6, 47.2, 70.5,...","[18.6, 16.9, 26.2, 38.2, 45.7, 52.5, 73.5, 89....","[38.4, 45.9, 46.5, 41.8, 49.0, 70.6, 101.8, 13..."
8,1629,2013-06-11 16:07:56,-8862.0,13342.6,12672.6,186797.0,101381.0,14,8,27.4,"[82.5, 85.0, 87.5, 90.0, 92.5, 95.0, 97.5, 100...","[-47.2, -35.9, -3.8, 15.7, 23.2, 32.6, 45.4, 5...","[37.9, 43.8, 30.6, 7.6, 4.5, 25.3, 55.2, 76.0,...","[60.5, 56.7, 30.8, 17.4, 23.7, 41.3, 71.5, 95...."
9,1719,2013-06-12 14:45:35,6603.0,6758.5,8234.0,27034.0,16468.0,4,2,28.9,"[82.5, 85.0, 87.5, 90.0, 92.5, 95.0, 97.5, 100...","[-78.8, -79.2, -37.8, 16.9, 43.5, 44.4, 29.3, ...","[-22.7, -28.0, -3.3, 33.7, 51.4, 47.7, 42.9, 4...","[82.0, 84.0, 37.9, 37.7, 67.4, 65.2, 52.0, 49...."
